In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
df_train = pd.read_parquet(r'C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\pl_combined.parquet').sort_values('Date', ascending=True)
df_test = pd.read_parquet(r'C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\season-2627.parquet').sort_values('Date', ascending=True)
df = pd.concat([df_test, df_train], ignore_index=True).sort_values('Date', ascending=True)

In [16]:
df2 = df.copy()
home_point_map = {'H': 3, 'D': 1, 'A': 0}
away_point_map = {'H': 0, 'D': 1, 'A': 3}
df2['HomePoint'] = df2['FTR'].map(home_point_map)
df2['AwayPoint'] = df2['FTR'].map(away_point_map)

In [17]:
df_home_history = df2[['Date', 'HomeTeam', 'FTHG', 'FTAG', 'HomePoint']].rename(
        columns={'HomeTeam': 'Team', 'FTHG': 'GoalsScored', 'FTAG': 'GoalsConceded', 'HomePoint': 'Point'}
    ).sort_values(['Team', 'Date']).reset_index(drop=True)
df_away_history = df2[['Date', 'AwayTeam', 'FTAG', 'FTHG', 'AwayPoint']].rename(
        columns={'AwayTeam': 'Team', 'FTAG': 'GoalsScored', 'FTHG': 'GoalsConceded', 'AwayPoint': 'Point'}
    ).sort_values(['Team', 'Date']).reset_index(drop=True)
df_home_history['Side'] = 'Home'
df_away_history['Side'] = 'Away'
df_team_history = pd.concat([df_home_history, df_away_history], ignore_index=True).sort_values(['Team', 'Date']).reset_index(drop=True)

In [18]:
for col, new_col in [
    ('GoalsScored', 'goals_avg_last_5'),
    ('GoalsConceded', 'conceded_avg_last_5'),
    ('Point', 'points_avg_last_5'),
]:
    df_team_history[new_col] = (
        df_team_history.groupby('Team')[col]
        .transform(lambda x: x.shift(1).rolling(5).mean())
    )

for col, new_col in [
    ('GoalsScored', 'home_goals_avg_home_last_5'),
    ('GoalsConceded', 'home_conceded_avg_home_last_5')
]:
    df_home_history[new_col] = (
        df_home_history.groupby('Team')[col]
        .transform(lambda x: x.shift(1).rolling(5).mean())
    )

for col, new_col in [
    ('GoalsScored', 'away_goals_avg_away_last_5'),
    ('GoalsConceded', 'away_conceded_avg_away_last_5')
]:
    df_away_history[new_col] = (
        df_away_history.groupby('Team')[col]
        .transform(lambda x: x.shift(1).rolling(5).mean())
    )

In [19]:
df2 = pd.merge(
    df2,
    df_team_history[(df_team_history['Side'] == 'Home')][
        ['Date', 'Team', 'goals_avg_last_5', 'conceded_avg_last_5', 'points_avg_last_5']
    ],
    left_on=['Date', 'HomeTeam'],
    right_on=['Date', 'Team'],
    how='left',
).drop(columns=['Team'])
df2 = df2.rename(
    columns={
        'goals_avg_last_5': 'home_goals_avg_last_5',
        'conceded_avg_last_5': 'home_conceded_avg_last_5',
        'points_avg_last_5': 'home_points_avg_last_5',
    }
)

df2 = pd.merge(
    df2,
    df_team_history[(df_team_history['Side'] == 'Away')][
        ['Date', 'Team', 'goals_avg_last_5', 'conceded_avg_last_5', 'points_avg_last_5']
    ],
    left_on=['Date', 'AwayTeam'],
    right_on=['Date', 'Team'],
    how='left',
).drop(columns=['Team'])
df2 = df2.rename(
    columns={
        'goals_avg_last_5': 'away_goals_avg_last_5',
        'conceded_avg_last_5': 'away_conceded_avg_last_5',
        'points_avg_last_5': 'away_points_avg_last_5',
    }
)

df2 = pd.merge(
    df2,
    df_home_history[
        ['Date', 'Team', 'home_goals_avg_home_last_5', 'home_conceded_avg_home_last_5']
    ],
    left_on=['Date', 'HomeTeam'],
    right_on=['Date', 'Team'],
    how='left',
).drop(columns=['Team'])

df2 = pd.merge(
    df2,
    df_away_history[
        ['Date', 'Team', 'away_goals_avg_away_last_5', 'away_conceded_avg_away_last_5']
    ],
    left_on=['Date', 'AwayTeam'],
    right_on=['Date', 'Team'],
    how='left',
).drop(columns=['Team'])

df2 = df2.drop(columns=['Matchday'])

In [23]:
train_df = df2[
    df2["season"].between("11/12", "24/25")
].copy()

valid_df = df2[df2["season"] == "25/26"].copy()
test_df = df2[df2["season"] == "26/27"].copy()

In [24]:
categorical_features = ["HomeTeam","AwayTeam"]
numerical_features = [
    "home_goals_avg_last_5",
    "home_conceded_avg_last_5",
    "home_points_avg_last_5",
    "away_goals_avg_last_5",
    "away_conceded_avg_last_5",
    "away_points_avg_last_5",
    "home_goals_avg_home_last_5",
    "home_conceded_avg_home_last_5",
    "away_goals_avg_away_last_5",
    "away_conceded_avg_away_last_5"
]
target_features = ["FTHG", "FTAG"]

X_train = train_df[categorical_features + numerical_features]
y_train = train_df[target_features]

X_valid = valid_df[categorical_features + numerical_features]
y_valid = valid_df[target_features]

X_test = test_df[categorical_features + numerical_features]
y_test = test_df[target_features]

In [25]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_cat = pd.DataFrame(
    encoder.fit_transform(X_train[categorical_features]),
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_train.index
)
X_valid_cat = pd.DataFrame(
    encoder.transform(X_valid[categorical_features]),
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_valid.index
)
X_test_cat = pd.DataFrame(
    encoder.transform(X_test[categorical_features]),
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_test.index
)

X_train_encoded = pd.concat(
    [X_train.drop(columns=categorical_features), X_train_cat],
    axis=1
)
X_valid_encoded = pd.concat(
    [X_valid.drop(columns=categorical_features), X_valid_cat],
    axis=1
)
X_test_encoded = pd.concat(
    [X_test.drop(columns=categorical_features), X_test_cat],
    axis=1
)

In [27]:
X_train_encoded.to_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_train_encoded.parquet",engine="pyarrow",compression="snappy",index=False,)
X_valid_encoded.to_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_valid_encoded.parquet",engine="pyarrow",compression="snappy",index=False,)
X_test_encoded.to_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_test_encoded.parquet",engine="pyarrow",compression="snappy",index=False,)
y_train.to_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\y_train.parquet",engine="pyarrow",compression="snappy",index=False,)
y_valid.to_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\y_valid.parquet",engine="pyarrow",compression="snappy",index=False,)
y_test.to_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\y_test.parquet",engine="pyarrow",compression="snappy",index=False,)
